  1. Load a representative subset of documents (e.g., 50–100 docs across diverse sources: Slack, Gmail, Jira, Confluence) and questions from EnterpriseRAG-Bench.                         
  2. Phase 1 (Data Cleaning & Normalization):                                                                                                                                             
      • Unicode NFKC normalization and whitespace cleanup.                                                                                                                                
      • Source-specific cleaning (e.g., Slack entity tag handling, email quote stripping).                                                                                                
      • Exact deduplication (SHA-256) check.                                                                                                                                              
  3. Phase 2 (Chunking):                                                                                                                                                                  
      • Parent-child hierarchical split (or recursive chunking) attaching metadata (doc_id, source_type).                                                                                 
  4. Phase 3 & 5 (Indexing & Hybrid Retrieval):                                                                                                                                           
      • Dense vector embeddings + Sparse BM25 index.                                                                                                                                      
      • Hybrid search test combining both with RRF.                                                                                                                                       
  5. Phase 6 (Re-Ranking):                                                                                                                                                                
      • Scoring top candidates using a cross-encoder model.                                                                                                                               
  6. Phase 7 (Generation & Prompting):                                                                                                                                                    
      • Formatting context with citations ([Doc ID]) and testing the output with an LLM.                                                                                                  
                                                                                                 

In [1]:
import os
from dotenv import load_dotenv
load_dotenv('/home/mahavir/code/python_projects/rag_benchmark/.env.dev')

hugguing_face_token = os.getenv('HUGGINGFACE_TOKEN')


# Phase 1. Data Loading

In [2]:
from datasets import load_dataset
DATASET_NAME = "ingalepratap/EnterpriseRAG-Bench"

/home/mahavir/code/python_projects/rag_benchmark/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
raw_dataset_questions = load_dataset(DATASET_NAME,'questions')
raw_dataset_questions

DatasetDict({
    test: Dataset({
        features: ['question_id', 'question_type', 'source_types', 'question', 'expected_doc_ids', 'gold_answer', 'answer_facts'],
        num_rows: 500
    })
})

In [4]:
raw_dataset_documents = load_dataset(DATASET_NAME,'documents')
raw_dataset_documents

DatasetDict({
    test: Dataset({
        features: ['doc_id', 'source_type', 'title', 'content'],
        num_rows: 511962
    })
})

Step 1: Data Cleaning ( Basic Cleaning & Normalization)

In [5]:
import unicodedata
import re

In [6]:
def normalize_text_for_rag(text:str)->str:
    
    # unicode NFKC Normalization (fixes ligatures,full width charas,accents)
    text = unicodedata.normalize("NFKC",text)

    # unify line breaks to standard unix format(\r\r and \r->\n)
    text = text.replace('\r\n','\n').replace("\r", "\n")

    # replace non breaking space and zero width spaces with stanard space
    # \u00a0 = non-breaking space, \u200b = zero-width space, \u202f = narrow no-break space
    text = re.sub(r"[\u00a0\u2007\xa0]"," ",text)
    text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)  # Strip zero-width markers

    # Step 4: Stitch hyphenated words broken across line wraps
    # Example: 'multi-\nlayer' -> 'multilayer'
    text = re.sub(r"(\b[a-zA-Z]+)-\n([a-zA-Z]+\b)", r"\1\2", text)

    # Step 5: Strip unprintable control characters (keep \n and \t)
    # Regex keeps standard ASCII, unicode letters/numbers/symbols, and basic whitespace
    text = "".join(
        ch for ch in text 
        if unicodedata.category(ch)[0] != "C" or ch in ("\n", "\t")
    )

    # Step 6: Normalize horizontal whitespace (spaces and tabs -> single space)
    text = re.sub(r"[ \t]+", " ", text)

    # Step 7: Collapse redundant blank lines (allow max 2 newlines for paragraph boundaries)
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Step 8: Strip leading and trailing whitespace per line
    text = "\n".join(line.strip() for line in text.split("\n"))

    return text.strip()


In [7]:
sample_raw_text = (
    "The ﬁnancial state\u00a0ment for Q3 shows a 25% in-\n"
    "crease in revenue.\r\n\r\n\r\n"
    "Full-width text: Ｔｅｓｔ Ｃａｓｅ.\n"
    "Ligatures test: ﬂow, aﬃliate.\n"
    "Erratic    spacing\t\tand   tabs."
)

In [8]:
raw_dataset_documents['test'][0]

{'doc_id': 'dsid_e54ef48bae78474684a957cf613d47d5',
 'source_type': 'confluence',
 'title': 'Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod)',
 'content': '## Purpose\nThis runbook describes the operational procedures to deploy, upgrade, roll back, and safely disable the **perf-canary** service across regions.\n\nperf-canary is a lightweight, always-on synthetic workload that calls internal inference endpoints for a curated model set and emits performance metrics (p50/p95/p99, tokens/sec, prefill/decode, queueing time, batch stats). The service must stay under the defined overhead budget and must not capture or emit customer data.\n\n**Primary users:** Eng-Infra on-call and Release Engineering during rollouts.\n\n---\n\n## Quick reference\n**Service name:** perf-canary\n\n**Deployment mechanism:**\n- Terraform: service account / IAM / secrets / region wiring\n- Helm: workload deployment and per-region config\n\n**Primary dashboards:**\n- Grafana: "Perf Canary – Per Model" (p95

In [9]:
raw_dataset_questions['test'][0]

{'question_id': 'qst_0001',
 'question_type': 'basic',
 'source_types': ['github'],
 'question': 'What are the default size limits for file uploads and total request size for the new multipart upload support on the OpenAI-compatible API endpoints?',
 'expected_doc_ids': ['dsid_ae068ee4aa9640159427cd941bef0238'],
 'gold_answer': 'The default limits are 10 MiB per file (max_file_size) and 50 MiB total per request (max_total_request_size) for multipart uploads on the OpenAI-compatible endpoints.',
 'answer_facts': ['The default per file upload size limit (max_file_size) for multipart uploads on OpenAI-compatible endpoints is 10 MiB.',
  'The default total request size limit (max_total_request_size) for multipart uploads on OpenAI-compatible endpoints is 50 MiB.']}

In [10]:
raw_dataset_documents['test'][0]

{'doc_id': 'dsid_e54ef48bae78474684a957cf613d47d5',
 'source_type': 'confluence',
 'title': 'Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod)',
 'content': '## Purpose\nThis runbook describes the operational procedures to deploy, upgrade, roll back, and safely disable the **perf-canary** service across regions.\n\nperf-canary is a lightweight, always-on synthetic workload that calls internal inference endpoints for a curated model set and emits performance metrics (p50/p95/p99, tokens/sec, prefill/decode, queueing time, batch stats). The service must stay under the defined overhead budget and must not capture or emit customer data.\n\n**Primary users:** Eng-Infra on-call and Release Engineering during rollouts.\n\n---\n\n## Quick reference\n**Service name:** perf-canary\n\n**Deployment mechanism:**\n- Terraform: service account / IAM / secrets / region wiring\n- Helm: workload deployment and per-region config\n\n**Primary dashboards:**\n- Grafana: "Perf Canary – Per Model" (p95

In [11]:
sample_q = raw_dataset_questions['test'][0]
normalize_sample_q = normalize_text_for_rag(sample_q['question'])
normalize_sample_q

'What are the default size limits for file uploads and total request size for the new multipart upload support on the OpenAI-compatible API endpoints?'

In [12]:
sample_d = raw_dataset_documents['test'][0]
normalize_sample_d_title = normalize_text_for_rag(sample_d['title'])
normalize_sample_d_content = normalize_text_for_rag(sample_d['content'])


In [13]:
def process_batch_questions(batch):
    return {'normalized_question': [normalize_text_for_rag(q) for q in batch["question"]]}

In [14]:
raw_dataset_questions= raw_dataset_questions.map(process_batch_questions,batched=True,num_proc=12)


Map (num_proc=12):   0%|          | 0/500 [00:00<?, ? examples/s]

Map (num_proc=12): 100%|██████████| 500/500 [00:00<00:00, 1955.69 examples/s]


In [15]:
raw_dataset_questions['test'][0]

{'question_id': 'qst_0001',
 'question_type': 'basic',
 'source_types': ['github'],
 'question': 'What are the default size limits for file uploads and total request size for the new multipart upload support on the OpenAI-compatible API endpoints?',
 'expected_doc_ids': ['dsid_ae068ee4aa9640159427cd941bef0238'],
 'gold_answer': 'The default limits are 10 MiB per file (max_file_size) and 50 MiB total per request (max_total_request_size) for multipart uploads on the OpenAI-compatible endpoints.',
 'answer_facts': ['The default per file upload size limit (max_file_size) for multipart uploads on OpenAI-compatible endpoints is 10 MiB.',
  'The default total request size limit (max_total_request_size) for multipart uploads on OpenAI-compatible endpoints is 50 MiB.'],
 'normalized_question': 'What are the default size limits for file uploads and total request size for the new multipart upload support on the OpenAI-compatible API endpoints?'}

In [16]:
raw_dataset_documents['test'][0]

{'doc_id': 'dsid_e54ef48bae78474684a957cf613d47d5',
 'source_type': 'confluence',
 'title': 'Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod)',
 'content': '## Purpose\nThis runbook describes the operational procedures to deploy, upgrade, roll back, and safely disable the **perf-canary** service across regions.\n\nperf-canary is a lightweight, always-on synthetic workload that calls internal inference endpoints for a curated model set and emits performance metrics (p50/p95/p99, tokens/sec, prefill/decode, queueing time, batch stats). The service must stay under the defined overhead budget and must not capture or emit customer data.\n\n**Primary users:** Eng-Infra on-call and Release Engineering during rollouts.\n\n---\n\n## Quick reference\n**Service name:** perf-canary\n\n**Deployment mechanism:**\n- Terraform: service account / IAM / secrets / region wiring\n- Helm: workload deployment and per-region config\n\n**Primary dashboards:**\n- Grafana: "Perf Canary – Per Model" (p95

In [17]:
def process_batch_documents(batch):
    batch['normalized_title'] = [ normalize_text_for_rag(t) for t in batch["title"]]
    batch['normalized_content'] = [ normalize_text_for_rag(t) for t in batch["content"]]
    return batch

In [18]:
raw_dataset_documents = raw_dataset_documents.map(process_batch_documents,batched=True,num_proc=12)

In [19]:
raw_dataset_documents['test'][0]

{'doc_id': 'dsid_e54ef48bae78474684a957cf613d47d5',
 'source_type': 'confluence',
 'title': 'Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod)',
 'content': '## Purpose\nThis runbook describes the operational procedures to deploy, upgrade, roll back, and safely disable the **perf-canary** service across regions.\n\nperf-canary is a lightweight, always-on synthetic workload that calls internal inference endpoints for a curated model set and emits performance metrics (p50/p95/p99, tokens/sec, prefill/decode, queueing time, batch stats). The service must stay under the defined overhead budget and must not capture or emit customer data.\n\n**Primary users:** Eng-Infra on-call and Release Engineering during rollouts.\n\n---\n\n## Quick reference\n**Service name:** perf-canary\n\n**Deployment mechanism:**\n- Terraform: service account / IAM / secrets / region wiring\n- Helm: workload deployment and per-region config\n\n**Primary dashboards:**\n- Grafana: "Perf Canary – Per Model" (p95

Step 2 Exact De Duplicaiton (SHA256)

In [20]:
import hashlib

In [21]:
def compute_sha256(text:str)->str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

In [22]:
seen_hashes = {}
duplicate_doc_ids = []

for doc in raw_dataset_documents['test']:
    doc_id = doc['doc_id']
    normalized_content = doc['normalized_content']
    content_hash = compute_sha256(normalized_content)

    if content_hash in seen_hashes:
        duplicate_doc_ids.append((doc_id,seen_hashes[content_hash]))
    else:
        seen_hashes[content_hash] = doc_id

In [23]:
print(f"Total documents: {len(raw_dataset_documents['test'])}")                                                                                                                       
print(f"Unique documents: {len(seen_hashes)}")                                                                                                                                        
print(f"Duplicate documents found: {len(duplicate_doc_ids)}")     

Total documents: 511962
Unique documents: 510390
Duplicate documents found: 1572


Step 3: Source-Specific Inspection & Cleaning     

In [24]:
source_types = set(raw_dataset_documents['test']['source_type'])                                                                                                                      
print("Available source types:", source_types)                                                                                                                                        
                                                                                                                                                                                          
# Inspect 1 sample from each source type                                                                                                                                              
samples_by_source = {}                                                                                                                                                                
for doc in raw_dataset_documents['test']:                                                                                                                                             
    st = doc['source_type']                                                                                                                                                           
    if st not in samples_by_source:                                                                                                                                                   
        samples_by_source[st] = doc                                                                                                                                                   
    if len(samples_by_source) == len(source_types):                                                                                                                                   
        break                                                                                                                                                                         
                                                                                                                                                                                          
for st, doc in samples_by_source.items():                                                                                                                                             
    print(f"\n=================== SOURCE: {st.upper()} ===================")                                                                                                          
    print("Title:", doc['title'])                                                                                                                                                     
    print("Content preview (first 250 chars):\n", doc['normalized_content'][:250])                                                                                                    
                                                                                                      

Available source types: {'github', 'fireflies', 'hubspot', 'gmail', 'slack', 'jira', 'google_drive', 'confluence', 'linear'}

=================== SOURCE: CONFLUENCE ===================
Title: Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod)
Content preview (first 250 chars):
 ## Purpose
This runbook describes the operational procedures to deploy, upgrade, roll back, and safely disable the **perf-canary** service across regions.

perf-canary is a lightweight, always-on synthetic workload that calls internal inference endpo

=================== SOURCE: FIREFLIES ===================
Title: Tethys Systems - SI Channel & Cloud GPU Fit Discovery
Content preview (first 250 chars):
 summary:
Introductory discovery between Redwood and Tethys Systems to evaluate a channel+SI partnership. Discussed integration options (hosted API vs. dedicated vs. private), expected customer use-cases (real-time chat, high-throughput batch embeddin

=================== SOURCE: GITHUB ===================
T

In [25]:
samples_by_source

{'confluence': {'doc_id': 'dsid_e54ef48bae78474684a957cf613d47d5',
  'source_type': 'confluence',
  'title': 'Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod)',
  'content': '## Purpose\nThis runbook describes the operational procedures to deploy, upgrade, roll back, and safely disable the **perf-canary** service across regions.\n\nperf-canary is a lightweight, always-on synthetic workload that calls internal inference endpoints for a curated model set and emits performance metrics (p50/p95/p99, tokens/sec, prefill/decode, queueing time, batch stats). The service must stay under the defined overhead budget and must not capture or emit customer data.\n\n**Primary users:** Eng-Infra on-call and Release Engineering during rollouts.\n\n---\n\n## Quick reference\n**Service name:** perf-canary\n\n**Deployment mechanism:**\n- Terraform: service account / IAM / secrets / region wiring\n- Helm: workload deployment and per-region config\n\n**Primary dashboards:**\n- Grafana: "Perf Canary

In [26]:
seen_hashes

{'d4ae11e7e6a7bc719bd30b7f0db7b8c127624989d5694657b3b558d5be228f8a': 'dsid_e54ef48bae78474684a957cf613d47d5',
 'a5ba217a4d3fe23454d5f1bfd825886609a47aed0d31a1568d585149c05803f6': 'dsid_229dd48e9b1d466a81ebaffe3ec84469',
 'c613a10be781dab69380235f32be5ac5c35c8695530e7ae28bbb2320f98d0e37': 'dsid_aeb0022d62bc43beb6549ba92e5655eb',
 'da1865e11ed2fb84d4e53b2a7dc98301b3cb339fc5102edb2143e58a3217b36c': 'dsid_926174fc4900408c89c98abde46b7225',
 '414c46db6cbc5b7b63c6d62e393aa256adddcc219afe36daf9128ef87ba12ca0': 'dsid_d511c6d8daf94f998ce6a3d97462af2d',
 '946193e38dd0b571e651001e0e3e4d26ba4c4d87578c41b0963baef4390604b6': 'dsid_321c06b3ec09434887168fcfdd458b26',
 '447db33682ca7e007bc5b157c0c71e532af561a89c7d7a464281dcf488011af5': 'dsid_a67150ff1ec94d0484d10954d223efae',
 '3f0221babe059ccfe4ca46e09d6460cfbb9dd4011c375793919b74178564d078': 'dsid_bddcc76fec5b4be3ae1c224904ae5786',
 '94b694fa2ffb9487d067d3b317b2f2cbb5671f41b6554e4912d3fedbbf167049': 'dsid_7e5d4cb344b44fe58a561b3759db09e6',
 '118c8cb3

In [27]:
duplicate_doc_ids

[('dsid_e8aabf8aad5b4ea79fd9d4ead0d3fb3f',
  'dsid_18f15cf5f684463399cf28f3c57c35e7'),
 ('dsid_d7fa0a27d51249279561abb57c793033',
  'dsid_c0853be552324961bb3d3c4457b15dd6'),
 ('dsid_9d2e4322f5e94b06b6de44d6cf484448',
  'dsid_f80581deb67e42c298bc7cd0fad2a0ea'),
 ('dsid_3745f56be1a04f5da3d780d99107adf0',
  'dsid_f51e85f51c6d4593ac1b549391223479'),
 ('dsid_ea82aceb4f7f424687b9fcac570db1df',
  'dsid_bbe4fa72c50045beb6746057bcb3d33e'),
 ('dsid_503be5e5473e4a57a2f8d6f2d649cffe',
  'dsid_11dd9890b5064bd69892b0b67aad62ee'),
 ('dsid_3078b4a35ddc415e8d1aa3d5a4b59a44',
  'dsid_0c15ff55d7e84225a19dd6b0db8e88d9'),
 ('dsid_0425be73d40d4cd49eb9d802eaab8007',
  'dsid_c3f3716189534aeb9df9f975478b6530'),
 ('dsid_be23061240384b99b3162d218424b7b7',
  'dsid_05eb5e0043c647419148dda5e51eb091'),
 ('dsid_9226161923b14532bc5f5f5b05dfadc2',
  'dsid_c6382ffd00e94ab780de29d9c6c06874'),
 ('dsid_6833aae7e0664bccb5d86efb9c59583a',
  'dsid_83f83eee6d4744fa90bd2508eaa59f4c'),
 ('dsid_675a3371196d4d22baad2bc90726f276',


In [28]:
seen_hashes

{'d4ae11e7e6a7bc719bd30b7f0db7b8c127624989d5694657b3b558d5be228f8a': 'dsid_e54ef48bae78474684a957cf613d47d5',
 'a5ba217a4d3fe23454d5f1bfd825886609a47aed0d31a1568d585149c05803f6': 'dsid_229dd48e9b1d466a81ebaffe3ec84469',
 'c613a10be781dab69380235f32be5ac5c35c8695530e7ae28bbb2320f98d0e37': 'dsid_aeb0022d62bc43beb6549ba92e5655eb',
 'da1865e11ed2fb84d4e53b2a7dc98301b3cb339fc5102edb2143e58a3217b36c': 'dsid_926174fc4900408c89c98abde46b7225',
 '414c46db6cbc5b7b63c6d62e393aa256adddcc219afe36daf9128ef87ba12ca0': 'dsid_d511c6d8daf94f998ce6a3d97462af2d',
 '946193e38dd0b571e651001e0e3e4d26ba4c4d87578c41b0963baef4390604b6': 'dsid_321c06b3ec09434887168fcfdd458b26',
 '447db33682ca7e007bc5b157c0c71e532af561a89c7d7a464281dcf488011af5': 'dsid_a67150ff1ec94d0484d10954d223efae',
 '3f0221babe059ccfe4ca46e09d6460cfbb9dd4011c375793919b74178564d078': 'dsid_bddcc76fec5b4be3ae1c224904ae5786',
 '94b694fa2ffb9487d067d3b317b2f2cbb5671f41b6554e4912d3fedbbf167049': 'dsid_7e5d4cb344b44fe58a561b3759db09e6',
 '118c8cb3

# Phase 2 (Chunking)

In [37]:
import uuid
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [50]:
subset_docs = raw_dataset_documents['test'].select(range(1000))

In [57]:
# 1. Define Splitters
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,
    chunk_overlap=300,
    separators=["\n\n", "\n", ". ", " "]
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " "]
)

# 2. Build Parent-Child Hierarchy with Enterprise Metadata Lineage
parent_store = {}  # Acts as the raw document store (e.g., Redis / MongoDB)
child_records = [] # Passed to Vector DB + BM25

for doc in subset_docs:
    doc_id = doc["doc_id"]
    source_type = doc["source_type"]
    title = doc.get("normalized_title", "")
    full_text = doc["normalized_content"]
    
    if not full_text.strip():
        continue

    # Generate Parent Chunks
    parents = parent_splitter.split_text(full_text)
    
    for p_idx, parent_text in enumerate(parents):
        parent_id = f"{doc_id}_p{p_idx}"
        
        # Save parent text with structural metadata
        parent_store[parent_id] = {
            "parent_id": parent_id,
            "doc_id": doc_id,
            "source_type": source_type,
            "title": title,
            "parent_text": parent_text
        }
        
        # Split parent into granular child chunks
        children = child_splitter.split_text(parent_text)
        for c_idx, child_text in enumerate(children):
            child_id = f"{parent_id}_c{c_idx}"
            child_records.append({
                "child_id": child_id,
                "parent_id": parent_id,
                "doc_id": doc_id,
                "source_type": source_type,
                "title": title,
                "child_text": child_text
            })

print(f"Ingested {len(subset_docs)} documents -> Generated {len(parent_store)} Parents, {len(child_records)} Child chunks.")

Ingested 1000 documents -> Generated 4119 Parents, 17771 Child chunks.


In [58]:
parent_store.keys()

dict_keys(['dsid_e54ef48bae78474684a957cf613d47d5_p0', 'dsid_e54ef48bae78474684a957cf613d47d5_p1', 'dsid_e54ef48bae78474684a957cf613d47d5_p2', 'dsid_e54ef48bae78474684a957cf613d47d5_p3', 'dsid_e54ef48bae78474684a957cf613d47d5_p4', 'dsid_229dd48e9b1d466a81ebaffe3ec84469_p0', 'dsid_229dd48e9b1d466a81ebaffe3ec84469_p1', 'dsid_229dd48e9b1d466a81ebaffe3ec84469_p2', 'dsid_229dd48e9b1d466a81ebaffe3ec84469_p3', 'dsid_aeb0022d62bc43beb6549ba92e5655eb_p0', 'dsid_aeb0022d62bc43beb6549ba92e5655eb_p1', 'dsid_aeb0022d62bc43beb6549ba92e5655eb_p2', 'dsid_aeb0022d62bc43beb6549ba92e5655eb_p3', 'dsid_926174fc4900408c89c98abde46b7225_p0', 'dsid_926174fc4900408c89c98abde46b7225_p1', 'dsid_926174fc4900408c89c98abde46b7225_p2', 'dsid_926174fc4900408c89c98abde46b7225_p3', 'dsid_926174fc4900408c89c98abde46b7225_p4', 'dsid_926174fc4900408c89c98abde46b7225_p5', 'dsid_926174fc4900408c89c98abde46b7225_p6', 'dsid_926174fc4900408c89c98abde46b7225_p7', 'dsid_d511c6d8daf94f998ce6a3d97462af2d_p0', 'dsid_d511c6d8daf94f9

In [59]:
parent_store['dsid_e54ef48bae78474684a957cf613d47d5_p0']

{'parent_id': 'dsid_e54ef48bae78474684a957cf613d47d5_p0',
 'doc_id': 'dsid_e54ef48bae78474684a957cf613d47d5',
 'source_type': 'confluence',
 'title': 'Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod)',
 'parent_text': '## Purpose\nThis runbook describes the operational procedures to deploy, upgrade, roll back, and safely disable the **perf-canary** service across regions.\n\nperf-canary is a lightweight, always-on synthetic workload that calls internal inference endpoints for a curated model set and emits performance metrics (p50/p95/p99, tokens/sec, prefill/decode, queueing time, batch stats). The service must stay under the defined overhead budget and must not capture or emit customer data.\n\n**Primary users:** Eng-Infra on-call and Release Engineering during rollouts.\n\n---\n\n## Quick reference\n**Service name:** perf-canary\n\n**Deployment mechanism:**\n- Terraform: service account / IAM / secrets / region wiring\n- Helm: workload deployment and per-region config\n\n**Pr

In [60]:
child_records

[{'child_id': 'dsid_e54ef48bae78474684a957cf613d47d5_p0_c0',
  'parent_id': 'dsid_e54ef48bae78474684a957cf613d47d5_p0',
  'doc_id': 'dsid_e54ef48bae78474684a957cf613d47d5',
  'source_type': 'confluence',
  'title': 'Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod)',
  'child_text': '## Purpose\nThis runbook describes the operational procedures to deploy, upgrade, roll back, and safely disable the **perf-canary** service across regions.\n\nperf-canary is a lightweight, always-on synthetic workload that calls internal inference endpoints for a curated model set and emits performance metrics (p50/p95/p99, tokens/sec, prefill/decode, queueing time, batch stats). The service must stay under the defined overhead budget and must not capture or emit customer data.\n\n**Primary users:** Eng-Infra on-call and Release Engineering during rollouts.\n\n---\n\n## Quick reference\n**Service name:** perf-canary\n\n**Deployment mechanism:**\n- Terraform: service account / IAM / secrets / region 

# Phase 3 - Indexing and Storing

In [62]:
child_records[0]

{'child_id': 'dsid_e54ef48bae78474684a957cf613d47d5_p0_c0',
 'parent_id': 'dsid_e54ef48bae78474684a957cf613d47d5_p0',
 'doc_id': 'dsid_e54ef48bae78474684a957cf613d47d5',
 'source_type': 'confluence',
 'title': 'Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod)',
 'child_text': '## Purpose\nThis runbook describes the operational procedures to deploy, upgrade, roll back, and safely disable the **perf-canary** service across regions.\n\nperf-canary is a lightweight, always-on synthetic workload that calls internal inference endpoints for a curated model set and emits performance metrics (p50/p95/p99, tokens/sec, prefill/decode, queueing time, batch stats). The service must stay under the defined overhead budget and must not capture or emit customer data.\n\n**Primary users:** Eng-Infra on-call and Release Engineering during rollouts.\n\n---\n\n## Quick reference\n**Service name:** perf-canary\n\n**Deployment mechanism:**\n- Terraform: service account / IAM / secrets / region wiring

In [63]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [record['child_text'].lower().split() for record in child_records]
bm25_index = BM25Okapi(tokenized_corpus)

print(f"BM25 Index built over {len(tokenized_corpus)} chunks.") 

BM25 Index built over 17771 chunks.


In [73]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [74]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1314.68it/s]


In [75]:
child_texts = [r["child_text"] for r in child_records]

In [76]:
child_embeddings = embedding_model.encode(
    child_texts,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches: 100%|██████████| 139/139 [00:36<00:00,  3.78it/s]


In [78]:
child_embeddings.shape

(17771, 384)

In [79]:
test_query = "perf-canary rollback procedure"                                                                                                                                         
                                                                                                                                                                                          
# 2. Tokenize the query exactly like the corpus (lowercase + split)                                                                                                                   
tokenized_query = test_query.lower().split()                                                                                                                                          
                                                                                                                                                                                          
# 3. Get BM25 scores across all 17,771 chunks                                                                                                                                         
bm25_scores = bm25_index.get_scores(tokenized_query)                                                                                                                                  
                                                                                                                                                                                          
# 4. Get the Top 3 highest scoring chunk indices                                                                                                                                      
top_indices = np.argsort(bm25_scores)[::-1][:3]                                                                                                                                       
                                                                                                                                                                                          
print(f"=== TOP 3 BM25 HITS FOR: '{test_query}' ===")                                                                                                                                 
for rank, idx in enumerate(top_indices, 1):                                                                                                                                           
    chunk = child_records[idx]                                                                                                                                                        
    print(f"\n[Rank {rank}] Score: {bm25_scores[idx]:.4f} | Source: {chunk['source_type'].upper()}")                                                                                  
    print(f"Title: {chunk['title']}")                                                                                                                                                 
    print(f"Parent ID: {chunk['parent_id']}")                                                                                                                                         
    print(f"Content: {chunk['child_text'][:250]}...")   

=== TOP 3 BM25 HITS FOR: 'perf-canary rollback procedure' ===

[Rank 1] Score: 17.1527 | Source: CONFLUENCE
Title: Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod)
Parent ID: dsid_e54ef48bae78474684a957cf613d47d5_p2
Content: ---

## Procedure C — Rollback (Helm)
Use when a perf-canary deploy/upgrade causes instability, overhead spikes, metrics breakage, or alert storms attributable to canary itself.

### C1. Decide rollback vs disable
- If perf-canary is **functioning bu...

[Rank 2] Score: 11.5577 | Source: CONFLUENCE
Title: Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod)
Parent ID: dsid_e54ef48bae78474684a957cf613d47d5_p1
Content: ### A6. Post-enable monitoring checklist
For the next 24 hours:
- Watch for noisy alerts (flapping) and ingestion lag.
- Confirm baseline lookups are succeeding (no “baseline missing” warnings).

---

## Procedure B — Standard upgrade (same region)
E...

[Rank 3] Score: 11.3324 | Source: CONFLUENCE
Title: Runbook: Deploy / Upgrade / Roll Ba

# Phase 4 - Retrivel(hubrid Search)

In [84]:
def hybrid_search(query:str,top_k:int = 50,rrf_k=60,candidate_pool:int = 50):

    # Step 1: Sparse Search
    tokenized_query = query.lower().split()
    bm25_scores = bm25_index.get_scores(tokenized_query)
    top_bm25_indices = np.argsort(bm25_scores)[::-1][:candidate_pool]

    # Step 2: Dense search 
    query_vector = embedding_model.encode(query,normalize_embeddings=True)
    dense_scores = child_embeddings @  query_vector
    top_dense_indices = np.argsort(dense_scores)[::-1][:candidate_pool]

    # Calculate Reciprocal Rank Fusion(RRF)
    rrf_scores = {}

    for rank,idx in enumerate(top_bm25_indices,start=1):
        rrf_scores[idx] = rrf_scores.get(idx,0.0) + (1.0 / (rrf_k + rank))

    for rank,idx in enumerate(top_dense_indices,start=1):
        rrf_scores[idx] = rrf_scores.get(idx,0.0) + (1.0 / (rrf_k + rank))

    # sort top candidates by combined RRF Scores
    fused_indices = sorted(rrf_scores.keys(),key=lambda i: rrf_scores[i],reverse=True)[:top_k]

    #Parent COntext Hydration
    hydrated_results = []  
    for rank,idx in enumerate(fused_indices,start=1):
        child = child_records[idx]
        parent_id = child['parent_id']
        parent = parent_store[parent_id]

        hydrated_results.append({                                                                                                                                                     
                "rank": rank,                                                                                                                                                             
                "rrf_score": rrf_scores[idx],                                                                                                                                             
                "bm25_score": float(bm25_scores[idx]),                                                                                                                                    
                "dense_score": float(dense_scores[idx]),                                                                                                                                  
                "child_id": child["child_id"],                                                                                                                                            
                "parent_id": parent_id,                                                                                                                                                   
                "doc_id": child["doc_id"],                                                                                                                                                
                "source_type": child["source_type"],                                                                                                                                      
                "title": child["title"],
                "matched_child_text": child["child_text"],
                "hydrated_parent_text": parent["parent_text"]
            })
    return hydrated_results


In [85]:
# test query

test_query= "What is the procedure to revert a canary deployment if it breaks?"

results = hybrid_search(test_query)
results

[{'rank': 1,
  'rrf_score': 0.029571646010002173,
  'bm25_score': 16.36940341942883,
  'dense_score': 0.5777729749679565,
  'child_id': 'dsid_b59df78d22b340309e1198654bcd7ffe_p5_c1',
  'parent_id': 'dsid_b59df78d22b340309e1198654bcd7ffe_p5',
  'doc_id': 'dsid_b59df78d22b340309e1198654bcd7ffe',
  'source_type': 'confluence',
  'title': 'Runbook: Cutting a Runtime LTS Patch Release (Serving Runtime)',
  'matched_child_text': 'Rollback procedure (high level):\n- Hosted/Dedicated: revert runtime image tag to previous patch; drain/rotate\n- Private: instruct customers to pin back to prior patch (provide exact tag + checksums)\n\nRM checklist (Step 10):\n- [ ] Rollout started with canary and monitored\n- [ ] Rollout completed or paused with documented rationale\n- [ ] Any rollback executed is communicated to Support/CS immediately\n\n---',
  'hydrated_parent_text': '### 10.2 Dedicated\n- Notify SRE + CS of rollout start window.\n- Roll out by customer tier/region according to change-control 

In [86]:
len(results)

50

In [87]:
results[0]

{'rank': 1,
 'rrf_score': 0.029571646010002173,
 'bm25_score': 16.36940341942883,
 'dense_score': 0.5777729749679565,
 'child_id': 'dsid_b59df78d22b340309e1198654bcd7ffe_p5_c1',
 'parent_id': 'dsid_b59df78d22b340309e1198654bcd7ffe_p5',
 'doc_id': 'dsid_b59df78d22b340309e1198654bcd7ffe',
 'source_type': 'confluence',
 'title': 'Runbook: Cutting a Runtime LTS Patch Release (Serving Runtime)',
 'matched_child_text': 'Rollback procedure (high level):\n- Hosted/Dedicated: revert runtime image tag to previous patch; drain/rotate\n- Private: instruct customers to pin back to prior patch (provide exact tag + checksums)\n\nRM checklist (Step 10):\n- [ ] Rollout started with canary and monitored\n- [ ] Rollout completed or paused with documented rationale\n- [ ] Any rollback executed is communicated to Support/CS immediately\n\n---',
 'hydrated_parent_text': '### 10.2 Dedicated\n- Notify SRE + CS of rollout start window.\n- Roll out by customer tier/region according to change-control agreements.

# Phase 5 - Re Ranking

In [88]:
from sentence_transformers import CrossEncoder

In [89]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
reranker

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 14144.91it/s]


CrossEncoder(
  (0): Transformer({'transformer_task': 'sequence-classification', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'logits'}}, 'module_output_name': 'scores', 'architecture': 'BertForSequenceClassification'})
)

In [91]:
def rerank_results(query: str, retrieved_candidates: list[dict], top_n: int = 3) -> list[dict]:                                                                                       
      """                                                                                                                                                                               
      Takes the Top 50 candidates from Hybrid Search and applies                                                                                                                        
      Cross-Encoder joint attention scoring to extract the Top N highest-precision docs.                                                                                                
      """                                                                                                                                                                               
      # 1. Prepare (Query, Document) pairs for the cross-encoder                                                                                                                        
      # Prepending the title gives the reranker vital domain context!                                                                                                                   
      pairs = [                                                                                                                                                                         
          [query, f"{cand['title']}\n{cand['matched_child_text']}"]                                                                                                                     
          for cand in retrieved_candidates
      ]                                                                                                                                                                                 
                                                                                                                                                                                        
      # 2. Predict cross-attention relevance logits                                                                                                                                     
      rerank_scores = reranker.predict(pairs)                                                                                                                                           
                                                                                                                                                                                        
      # 3. Attach rerank scores to candidates                                                                                                                                           
      for i, cand in enumerate(retrieved_candidates):                                                                                                                                   
          cand["rerank_score"] = float(rerank_scores[i])                                                                                                                                
                                                                                                                                                                                        
      # 4. Sort candidates by cross-encoder score descending                                                                                                                            
      reranked_candidates = sorted(retrieved_candidates, key=lambda x: x["rerank_score"], reverse=True)                                                                                 
                                                                                                                                                                                        
      # Re-assign final rank                                                                                                                                                            
      for final_rank, cand in enumerate(reranked_candidates[:top_n], start=1):
          cand["final_rank"] = final_rank
          
      return reranked_candidates[:top_n]

In [92]:
top_3_reranked = rerank_results(test_query, results, top_n=3)

In [93]:
top_3_reranked[0]

{'rank': 6,
 'rrf_score': 0.01639344262295082,
 'bm25_score': 11.88094912956195,
 'dense_score': 0.6206021308898926,
 'child_id': 'dsid_e54ef48bae78474684a957cf613d47d5_p2_c2',
 'parent_id': 'dsid_e54ef48bae78474684a957cf613d47d5_p2',
 'doc_id': 'dsid_e54ef48bae78474684a957cf613d47d5',
 'source_type': 'confluence',
 'title': 'Runbook: Deploy / Upgrade / Roll Back perf-canary (Prod)',
 'matched_child_text': '---\n\n## Procedure C — Rollback (Helm)\nUse when a perf-canary deploy/upgrade causes instability, overhead spikes, metrics breakage, or alert storms attributable to canary itself.\n\n### C1. Decide rollback vs disable\n- If perf-canary is **functioning but noisy**: prefer to mute alerts / adjust config, not rollback.\n- If perf-canary is **malfunctioning** (crash loop, spamming endpoints, violating safety constraints): rollback immediately.\n- If safety/privacy is at risk: disable immediately (see “Safe-disable procedures”), then rollback.\n\n### C2. Execute rollback\n1) Identify l

# Phase: Final LLM Generation

In [95]:
def format_context_for_llm(reranked_docs: list[dict]) -> str:                                                                                                                         
    """                                                                                                                                                                               
    Assembles the hydrated parent documents into a structured context block                                                                                                           
    with explicit Document IDs for citations.                                                                                                                                         
    """                                                                                                                                                                               
    context_blocks = []                                                                                                                                                               
                                                                                                                                                                                         
    for i, doc in enumerate(reranked_docs, start=1):                                                                                                                                  
        block = (                                                                                                                                                                     
            f"--- DOCUMENT {i} ---\n"                                                                                                                                                 
            f"Doc ID: {doc['doc_id']}\n"                                                                                                                                              
            f"Source: {doc['source_type'].upper()}\n"                                                                                                                                 
            f"Title: {doc['title']}\n"                                                                                                                                                
            f"Content:\n{doc['hydrated_parent_text']}\n"                                                                                                                              
        )                                                                                                                                                                             
        context_blocks.append(block)                                                                                                                                                  
                                                                                                                                                                                         
    return "\n".join(context_blocks)                                                                                                                                                  
                                                                                                                                                                                         
formatted_context = format_context_for_llm(top_3_reranked)                                                                                                                            
print(f"Context assembled! Total characters: {len(formatted_context)}")       

Context assembled! Total characters: 8802


In [96]:
                                                                                                                                                                                      
SYSTEM_PROMPT = """You are an internal enterprise AI assistant for Redwood Inference.                                                                                                 
    Your task is to answer the user's question based strictly and exclusively on the provided internal documents.                                                                         
                                                                                                                                                                                          
    STRICT OPERATIONAL RULES:                                                                                                                                                             
    1. CITATIONS: Every single factual statement, step, or recommendation you make MUST include an inline citation with its Document ID, formatted as [Doc ID: dsid_xxx].                 
    2. ZERO HALLUCINATION / REFUSAL: If the provided documents do not contain sufficient evidence to answer the question, state: "Information not found in internal documents." Do NOT    
  extrapolate, speculate, or use outside knowledge.                                                                                                                                       
    3. CONFLICTS: If two documents provide contradictory procedures or numbers, explicitly state both viewpoints and cite their respective Document IDs.                                  
    4. TONE: Professional, concise, and structured (use bullet points and bold headers for operational procedures).                                                                       
    """                                                                                                                                                                                   
                                                                                                                                                                                          
def build_rag_prompt(query: str, context: str) -> str:                                                                                                                                
    return f"""{SYSTEM_PROMPT}                                                                                                                                                        
                                                                                                                                                                                          
CONTEXT DOCUMENTS:                                                                                                                                                                    
{context}                                                                                                                                                                             
                                                                                                                                                                                          
USER QUESTION:                                                                                                                                                                        
{query}                                                                                                                                                                               
                                                                                                                                                                                          
ANSWER:"""                

In [104]:
from transformers import pipeline                                                                                                                                                     
import torch                                                                                                                                                                          
                                                                                                                                                                                          
# 1. Detect GPU device directly without needing the 'accelerate' package                                                                                                              
device = "cuda" if torch.cuda.is_available() else "cpu"                                                                                                                               
print(f"Using compute device: {device}")   

generator = pipeline(                                                                                                                                                                 
        "text-generation",                                                                                                                                                                
        model="Qwen/Qwen2.5-0.5B-Instruct",  # Or "Qwen/Qwen2.5-7B-Instruct" if you have GPU VRAM                                                                                         
        device=device,                                                                                                                                                                
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32                                                                                                     
    )                                                                                                                                                                                     
                                                                                                                                                                                          
# 3. Format the chat prompt
messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"CONTEXT DOCUMENTS:\n{formatted_context}\n\nUSER QUESTION:\n{test_query}"}
    ]                                                                                                                                                                                
                                                                                                                                                                                          
# 4. Generate grounded answer                                                                                                                                                         
print("Generating grounded answer on GPU...")                                                                                                                                         
output = generator(                                                                                                                                                                   
        messages,                                                                                                                                                                         
        max_new_tokens=450,                                                                                                                                                               
        do_sample=False  # Deterministic greedy decoding for maximum factual accuracy                                                                                                     
    )                                                                                                                                                                                     
  
generated_answer = output[0]["generated_text"][-1]["content"]
print("\n=== FINAL GROUNDED RAG ANSWER ===\n")
print(generated_answer)

Using compute device: cuda


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 431.96it/s]


Generating grounded answer on GPU...

=== FINAL GROUNDED RAG ANSWER ===

To revert a canary deployment that breaks, you should follow these steps:

1. **Identify the Last Known Good Revision:** Use `helm history` to find the latest commit that marked the canary as being ready for rollback.

2. **Roll Back the Deployment:** Use `helm rollback` to apply the rollback process. This will revert the canary deployment to its previous state.

Here’s how you can implement this in your workflow:

### Procedure Steps:

1. **Pre-flight Checks:**
   - Identify the last known good revision using `helm history`.
   - Confirm the current version using `helm list`.

2. **Apply Rollback:**
   - Roll back the deployment using `helm rollback`.
   - Watch the rollout process to ensure all pods drain and replace cleanly.

3. **Post-rollback Tasks:**
   - Confirm alert rules are in the desired state (muted/unmuted).
   - Update `#eng-releases` and `#eng-runtime` with details about the rollback.
   - Close an

# 